In [ ]:
import pandas as pd
import seaborn as sea
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import contingency

# um Warnungen zu ignorieren (auf eigene Gefahr!)
import warnings
warnings.filterwarnings("ignore")

# limitierung der Anzeigegenauigkeit
pd.options.display.precision = 4

In [ ]:
# einladen der Daten
pilze = pd.read_excel('./data/mushroom-colors.xlsx', index_col=0)
display(pilze.head(3))

In [ ]:
pilze.info()

In [ ]:
# erstellen der Kreuztabelle mit pd.crosstab
# crosstab: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.crosstab.html
#ct = pd.crosstab(pilze['Hutfarbe'], pilze['Lamellenfarbe'], margins=True)
ct = pd.crosstab(pilze['Klasse'], pilze['Lamellenfarbe'], margins=True)

display(ct)

In [ ]:
# in normalisierter Form
ct_norm = pd.crosstab(pilze['Hutfarbe'], pilze['Lamellenfarbe'], margins=True, normalize=True)
display(ct_norm)

In [ ]:
# Randverteilungen mit jointplot
# jointplot: https://seaborn.pydata.org/generated/seaborn.jointplot.html
fig = sea.jointplot(data=pilze, x='Hutfarbe', y='Lamellenfarbe', kind='hist')
fig.ax_joint.set_xticklabels(fig.ax_joint.get_xticklabels(),rotation=45)
plt.show()

In [ ]:
# bedingte Häufigkeiten
# P(Lamellenfarbe=brown | Hutfarbe = brown)
hij = 348
hsj = 2284
print('P(Lamellenfarbe=brown | Hutfarbe = brown) : {0:.4f}'.format(hij/hsj))
df=pd.crosstab(pilze['Hutfarbe'], pilze['Lamellenfarbe'], margins=True, normalize='index')
display(df.loc['brown'])

In [ ]:
# P(Hutfarbe = brown | Lamellenfarbe=brown)
hij = 348
hsj = 1048
print('P(Hutfarbe = brown | Lamellenfarbe=brown) : {0:.4f}'.format(hij/hsj))
df=pd.crosstab(pilze['Hutfarbe'], pilze['Lamellenfarbe'], margins=True, normalize='columns')
display(df.loc[:'brown'])

## Frage

Bestimme die Kreuztabelle für die Merkmale *Hüllenfarbe* und *Klasse*. In der Kreuztabelle sollen die absoluten Randhäufigkeiten ablesbar sein. Wie lauten die bedingten Häufigkeiten für:
- P(Klasse=edible | Hüllenfarbe=orange)
- P(Hüllenfarbe=yellow | Klasse=poisonous)
- P(Hüllenfarbe=white | Klasse=poisonous)
- P(Klasse=poisonous | Hüllenfarbe=white)

In [ ]:
crosstab = pd.crosstab(pilze['Hüllenfarbe'], pilze['Klasse'], margins=False, normalize=True)
crosstab2 = pd.crosstab(pilze['Klasse'], pilze['Hüllenfarbe'], margins=False, normalize=True)

#display(crosstab)

print("P(Klasse=edible | Hüllenfarbe=orange:", crosstab2.loc['edible', 'orange'])
print("P(Hüllenfarbe=yellow | Klasse=poisonous:", crosstab.loc['yellow', 'poisonous'])
print("P(Hüllenfarbe=white | Klasse=poisonous:", crosstab.loc['white', 'poisonous'])
print("P(Klasse=poisonous | Hüllenfarbe=white:", crosstab2.loc['poisonous', 'white'])




## Kontingenzkoeffizient

In [ ]:
# einfaches Beispiel
df = pilze[(pilze['Hüllenfarbe'] == 'yellow') | (pilze['Hüllenfarbe'] == 'orange')]
ktab = pd.crosstab(df['Hüllenfarbe'], df['Klasse'], margins=False)
display(ktab)

# Bestimmung der statistischen Abhängigkeit
# chi2_contingency: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.chi2_contingency.html
chi2, pValue, dof, expFreq = contingency.chi2_contingency(ktab, correction=False)
print('Teststatistik (chi-quadrat): {0:.4f}'.format(chi2))
print('P-Wert: {0:.4f}'.format(pValue))
print('Freiheitsgrade: {0:d}'.format(dof))
print('erwartete Häufigkeiten')
expFreqDF = pd.DataFrame(expFreq, index=ktab.index, columns= ktab.columns)
display(expFreqDF)

# Korrektur
npoints = df.shape[0] # Anzahl an Beobachtungen
nExpr = min(ktab.shape) # minimale Anzahl an Ausprägungen bei den beiden Merkmalen
ub = npoints*(nExpr-1) # oberes limit
print('Obere Grenze für Chi-Quadrat: {0:.4f}'.format(ub))
# Wert für Kontingenzkoeffizient nach Pearson
print('Pearson: {0:.4f} (max. {1:.4f})'.format(np.sqrt(chi2/(npoints+chi2)),np.sqrt((nExpr-1)/nExpr)))
print('Pearson normalisiert: {0:.4f} (max. 1)'.format(np.sqrt(chi2/(npoints+chi2))*np.sqrt(nExpr/(nExpr-1))))

In [ ]:
# selbes problem aber zusätzlich mit Pilzen mit weißer Hüllenfarbe
df = pilze[(pilze['Hüllenfarbe'] == 'yellow') | (pilze['Hüllenfarbe'] == 'orange') | (pilze['Hüllenfarbe'] == 'white')]
ktab = pd.crosstab(df['Hüllenfarbe'], df['Klasse'], margins=False)
display(ktab)

# Bestimmung der statistischen Abhängigkeit
chi2, pValue, dof, expFreq = contingency.chi2_contingency(ktab)
print('Teststatistik (chi-quadrat): {0:.4f}'.format(chi2))
print('P-Wert: {0:.4f}'.format(pValue))
print('Freiheitsgrade: {0:d}'.format(dof))
print('erwartete Häufigkeiten')
expFreqDF = pd.DataFrame(expFreq, index=ktab.index, columns= ktab.columns)
display(expFreqDF)

# Korrektur
npoints = df.shape[0] # Anzahl an Beobachtungen
nExpr = min(ktab.shape) # minimale Anzahl an Ausprägungen bei den beiden Merkmalen
ub = npoints*(nExpr-1) # oberes limit
print('Obere Grenze für Chi-Quadrat: {0:.4f}'.format(ub))
# Wert für Kontingenzkoeffizient nach Pearson
print('Pearson: {0:.4f} (max. {1:.4f})'.format(np.sqrt(chi2/(npoints+chi2)),np.sqrt((nExpr-1)/nExpr)))
print('Pearson korrigiert: {0:.4f} (max. 1)'.format(np.sqrt(chi2/(npoints+chi2))*np.sqrt(nExpr/(nExpr-1))))

In [ ]:
# selbes problem aber zusätzlich mit Pilzen mit weißer Hüllenfarbe
df = pilze
ktab = pd.crosstab(df['Hutfarbe'], df['Klasse'], margins=False)
display(ktab)

# Bestimmung der statistischen Abhängigkeit
chi2, pValue, dof, expFreq = contingency.chi2_contingency(ktab)
print('Teststatistik (chi-quadrat): {0:.4f}'.format(chi2))
print('P-Wert: {0:.4f}'.format(pValue))
print('Freiheitsgrade: {0:d}'.format(dof))
print('erwartete Häufigkeiten')
expFreqDF = pd.DataFrame(expFreq, index=ktab.index, columns= ktab.columns)
display(expFreqDF)

# Korrektur
npoints = df.shape[0] # Anzahl an Beobachtungen
nExpr = min(ktab.shape) # minimale Anzahl an Ausprägungen bei den beiden Merkmalen
ub = npoints*(nExpr-1) # oberes limit
print('Obere Grenze für Chi-Quadrat: {0:.4f}'.format(ub))
# Wert für Kontingenzkoeffizient nach Pearson
print('Pearson: {0:.4f} (max. {1:.4f})'.format(np.sqrt(chi2/(npoints+chi2)),np.sqrt((nExpr-1)/nExpr)))
print('Pearson korrigiert: {0:.4f} (max. 1)'.format(np.sqrt(chi2/(npoints+chi2))*np.sqrt(nExpr/(nExpr-1))))

# kann auch mittels contingency.association(ktab, method='pearson') ermittelt werden
pearson = contingency.association(ktab, method='pearson')
print('mit association-Methode, Pearson: {0:.4f}'.format(pearson))
print('mit association-Methode, Pearson korrigiert: {0:.4f} (max. 1)'.format(pearson*np.sqrt(nExpr/(nExpr-1))))

## Frage

Berechne und interpretiere den korrigierten Kontingenzkoeffizienten nach Pearson für folgende Merkmalskombinationen:
- Klasse und Hutfarbe
- Lamellen- und Sporenpulverfarbe
  

In [35]:
crosstab = pd.crosstab(pilze['Klasse'], pilze['Hutfarbe'], margins=False)
display(crosstab)

chi2, pValue, dof, expFreq = contingency.chi2_contingency(crosstab)
nPoints = pilze.shape[0]
nExpr = min(crosstab.shape)

print('Pearson korrigiert: {0:.4f} (max. 1)'.format(np.sqrt(chi2/(nPoints+chi2))*np.sqrt(nExpr/(nExpr-1))))

crosstab = pd.crosstab(pilze['Lamellenfarbe'], pilze['Sporenpulverfarbe'], margins=False)
display(crosstab)

chi2, pValue, dof, expFreq = contingency.chi2_contingency(crosstab)
nPoints = pilze.shape[0]
nExpr = min(crosstab.shape)

print('Pearson korrigiert: {0:.4f} (max. 1)'.format(np.sqrt(chi2/(nPoints+chi2))*np.sqrt(nExpr/(nExpr-1))))



Hutfarbe,brown,buff,cinnamon,gray,green,pink,purple,red,white,yellow
Klasse,,,,,,,,,,
edible,1264,48,32,1032,16,56,16,624,720,400
poisonous,1020,120,12,808,0,88,0,876,320,672


Pearson korrigiert: 0.3018 (max. 1)


Sporenpulverfarbe,black,brown,buff,chocolate,green,orange,purple,white,yellow
Lamellenfarbe,,,,,,,,,
black,204,204,0,0,0,0,0,0,0
brown,476,508,16,0,0,16,16,0,16
buff,0,0,0,0,0,0,0,1728,0
chocolate,96,96,0,540,0,0,0,0,0
gray,100,100,0,432,24,0,0,96,0
green,0,0,0,0,24,0,0,0,0
orange,0,16,16,0,0,16,0,0,16
pink,412,428,0,540,0,0,16,96,0
purple,240,240,0,12,0,0,0,0,0


Pearson korrigiert: 0.8572 (max. 1)


## Quantitative Daten

In [ ]:
# einladen des Pinguin-Datensatzes
penguins = pd.read_excel('./data/penguinsSmall.xlsx', index_col=0)

# pairplot als anschauliche Darstellung der Zusammenhänge
# pairplot: https://seaborn.pydata.org/generated/seaborn.pairplot.html
sea.pairplot(data = penguins, kind='scatter', corner=True, hue='Species', height=1.75)
plt.show()

## Frage

Erstelle einen pairplot mit:
- kde-Plots für die einzelnen Merkmalspaarungen
- Histogrammen entlang der Diagonalen
- Farbkodierung basierend auf dem Geschlecht

In [ ]:
sea.pairplot(data = penguins, kind='kde', corner=True, hue='Species', height=1.75)
sea.pairplot(data= penguins, kind='hist', corner=True, hue='Species', height=1.75)
sea.pairplot(data= penguins, kind='hist', corner=True, hue='Sex', height=1.75, diag_kind='hist')

## Frage

Berechne für die einzelnen Merkmalspaarungen die entsprechenden Kovarianz. Verwende dazu die Methode [np.cov()](https://numpy.org/doc/stable/reference/generated/numpy.cov.html). Bestimme anschließend die entsprechenden Korrelationen **rechnerisch**. Überprüfe dein Ergebnis mithilfe der [corr-Methode](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html) von pandas.

In [ ]:
df = penguins.groupby('Species')['Body Mass (g)']
df.corr(df)